# 10 · Tucker decomposition on real data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/10-tucker-decomposition.ipynb)

*Part IV · exercise · 15 min*

> 🇪🇸 **Descomposición de Tucker con datos reales** — PCA generalizado a todos los ejes, sobre un tensor real de viajes en taxi de Nueva York.

PCA generalized to every axis, on a real tensor of New York taxi trips.

## What you will be able to do

- Build a genuine order-3 tensor out of a flat table of real trips.
- Compute a Tucker decomposition by HOSVD, using only unfolding, SVD and einsum.
- Contract three axes at once with a single `einsum` string.
- Measure reconstruction error against compression ratio.
- Read a factor matrix and recognise a real pattern the decomposition found by itself.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from skimage import data

TAXIS = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/taxis.csv"
taxis = pd.read_csv(TAXIS)

def unfold(T, axis):
    return np.moveaxis(T, axis, 0).reshape(T.shape[axis], -1)

print(taxis.shape)                       # (6433, 14) — 6,433 real NYC taxi trips

## Rank you can see

> 🇪🇸 Antes de generalizar a tensores, comprimamos una sola matriz: una
> imagen real. La SVD truncada de rango k conserva las k direcciones
> singulares más fuertes y descarta el resto — por Eckart–Young, es la mejor
> aproximación de rango k posible en norma de Frobenius.

Before generalizing to tensors, let's compress a single matrix — a real
image. The rank-`k` truncated SVD keeps only the `k` strongest singular
directions and drops the rest. By the **Eckart–Young theorem**, that
truncation is the *optimal* rank-`k` approximation to the original matrix in
Frobenius norm — no other rank-`k` matrix is closer.

We'll reconstruct a 512×512 grayscale photograph (`skimage.data.camera()`) at
`k = 1, 5, 20, 50` and full rank, and compare three things side by side: how
much storage each reconstruction needs, how much of the image's Frobenius
energy it retains, and how it actually looks.

In [ ]:
img = data.camera().astype(float)
m, n = img.shape                          # (512, 512)

U, s, Vt = np.linalg.svd(img, full_matrices=False)

ks = [1, 5, 20, 50, min(m, n)]
total_energy = np.sum(s**2)

fig, axes = plt.subplots(1, len(ks), figsize=(15, 3.5))
for ax, k in zip(axes, ks):
    recon = (U[:, :k] * s[:k]) @ Vt[:k, :]
    stored = k * (m + n + 1)                          # mk + k + nk
    storage_pct = 100 * stored / (m * n)
    factor = (m * n) / stored
    energy_pct = 100 * np.sum(s[:k]**2) / total_energy
    label = "full rank" if k == min(m, n) else f"k={k}"
    ax.imshow(recon, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"{label}\n{storage_pct:.1f}% storage, {factor:.1f}x\n{energy_pct:.1f}% energy",
                 fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

for k in ks:
    stored = k * (m + n + 1)
    print(f"k={k:>3}  storage={100*stored/(m*n):6.2f}%  "
          f"{(m*n)/stored:6.2f}x  energy={100*np.sum(s[:k]**2)/total_energy:6.2f}%")
# k=  1  storage=  0.39%  255.75x  energy= 87.01%
# k=  5  storage=  1.96%   51.15x  energy= 97.04%
# k= 20  storage=  7.82%   12.79x  energy= 98.98%
# k= 50  storage= 19.55%    5.12x  energy= 99.60%
# k=512  storage=200.20%    0.50x  energy=100.00%

## Storage, energy, and what your eyes see

> 🇪🇸 El almacenamiento, la energía retenida y la calidad perceptual no son
> la misma curva. Con muy pocos componentes ya se retiene casi toda la
> energía, y la imagen es reconocible con una fracción minúscula del
> almacenamiento original. La misma idea — quedarse con las direcciones más
> fuertes y descartar el resto — es exactamente lo que Tucker/HOSVD hace a
> continuación, un eje del tensor a la vez.

At `k = 1`, under 0.4% of the storage already recovers 87% of the energy —
but the picture is barely recognisable. By `k = 20`, storage is still under
8% of the original and the picture is already unmistakably the photograph,
while the energy curve hasn't yet reached its final digit. At full rank, the
factorized `U`, `s`, `Vt` together need *more* numbers than the dense image
itself (about 200% of its storage) — factorizing only pays off once you
truncate. The same idea — keep the strongest singular directions, drop the
rest — is what Tucker/HOSVD does next, one tensor axis at a time.

**The picture is recognisable at `k = 20` — under 8% of the storage — long
before the numbers claim it should be. Energy retained and perceptual
quality are not the same curve.**

## The theory

> 🇪🇸 PCA comprime una **matriz**: dos ejes. La descomposición de Tucker
> generaliza PCA a un tensor de cualquier orden: una **matriz de factores por
> eje**, más un **tensor núcleo** pequeño.

PCA compresses a **matrix** — two axes. Real data often has more. **Tucker
decomposition** generalizes PCA to a tensor of any order: one **factor matrix
per axis**, plus a small **core tensor** describing how the factors combine.

The way to compute it, called **HOSVD**, uses only tools you already have:

1. **Unfold** the tensor along each axis (section 01).
2. Run **SVD** on each unfolding; keep the top components. These are the factor
   matrices.
3. **Contract** the original tensor against all factor matrices to get the core
   (section 06).

The related **CP decomposition** instead writes the tensor as a sum of simple
rank-1 pieces. Tucker is usually more accurate at the same size; CP is often
easier to interpret.

## Our real tensor

From 6,433 real New York taxi trips we build a genuine order-3 tensor:
**pickup borough × dropoff borough × hour of day.**

> 🇪🇸 Un tensor real de orden 3: barrio de origen × barrio de destino × hora.

In [ ]:
taxis['hour'] = pd.to_datetime(taxis['pickup']).dt.hour
sub = taxis.dropna(subset=['pickup_borough', 'dropoff_borough'])
pb = sorted(sub['pickup_borough'].unique())
db = sorted(sub['dropoff_borough'].unique())

T = np.zeros((len(pb), len(db), 24))
for (p, d, h), v in sub.groupby(['pickup_borough', 'dropoff_borough', 'hour']).size().items():
    T[pb.index(p), db.index(d), h] = v

print(T.shape, pb, db)

## Exercise 1 — read the tensor before you decompose it

> 🇪🇸 Entiende el tensor antes de descomponerlo.

In [ ]:
# TODO 1: Print T.shape and T.sum(). What does the entry T[i, j, k] mean?

# TODO 2: Which hour has the most trips overall? (Sum over the first two axes.)

# TODO 3: Unfold T along each axis and print the three shapes. Confirm the total
#         number of entries is the same each time — unfolding loses nothing.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
print(T.shape, T.sum())
# T[i, j, k] = how many trips started in borough pb[i], ended in borough db[j],
# and were picked up during hour k.

by_hour = T.sum(axis=(0, 1))
print(by_hour.argmax())                    # 18 — evening rush hour

for ax in range(3):
    M = unfold(T, ax)
    print(ax, M.shape, M.size == T.size)   # True every time

## Exercise 2 — HOSVD, in two einsum calls

> 🇪🇸 HOSVD en dos llamadas a einsum.

Look at the einsum strings you are about to write: `'ijk,ia,jb,kc->abc'`
contracts three axes in one expression. **That is why einsum came first.**

In [ ]:
# TODO 4: Run SVD on each unfolding, keep the top (2, 2, 3) components, and
#         build the core tensor with ONE einsum call.

# TODO 5: Reconstruct T from the core and factors, again with one einsum.
#         Compute the relative error and the compression ratio.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
Us = [np.linalg.svd(unfold(T, ax), full_matrices=False)[0] for ax in range(3)]
r = (2, 2, 3)
Us = [Us[i][:, :r[i]] for i in range(3)]
print([u.shape for u in Us])

core  = np.einsum('ijk,ia,jb,kc->abc', T, Us[0], Us[1], Us[2])   # (2, 2, 3)
recon = np.einsum('abc,ia,jb,kc->ijk', core, Us[0], Us[1], Us[2])

error = np.linalg.norm(T - recon) / np.linalg.norm(T)            # 0.067
ratio = T.size / (core.size + sum(u.size for u in Us))           # 4.71
print(core.shape, round(error, 3), round(ratio, 2))

The exercise above fixed one rank, (2, 2, 3). Move the slider to see the
whole error/compression trade-off, not just that one point on it.

> 🇪🇸 El ejercicio anterior fijó un solo rango, (2, 2, 3). Mueve el
> deslizador para ver toda la curva de compensación, no solo ese punto.

In [ ]:
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import ipywidgets as widgets
import matplotlib.pyplot as plt

# Precompute the full SVD basis for each axis once; the slider only re-slices
# and re-contracts these small matrices, which is what keeps it responsive.
bases = [np.linalg.svd(unfold(T, ax), full_matrices=False)[0] for ax in range(3)]
max_rank = min(u.shape[1] for u in bases)

ks, errors, ratios = list(range(1, max_rank + 1)), [], []
for kk in ks:
    Uk = [bases[ax][:, :kk] for ax in range(3)]
    core_k = np.einsum('ijk,ia,jb,kc->abc', T, *Uk)
    recon_k = np.einsum('abc,ia,jb,kc->ijk', core_k, *Uk)
    errors.append(np.linalg.norm(T - recon_k) / np.linalg.norm(T))
    ratios.append(T.size / (core_k.size + sum(u.size for u in Uk)))

def show_rank(k):
    i = k - 1
    plt.close('all')
    fig, ax1 = plt.subplots(figsize=(6, 3.2))
    ax1.plot(ks, errors, color='#C44E52')
    ax1.scatter([k], [errors[i]], color='#C44E52', zorder=5)
    ax1.set_xlabel('rank k (shared across all three axes)')
    ax1.set_ylabel('relative error', color='#C44E52')
    ax2 = ax1.twinx()
    ax2.plot(ks, ratios, color='#4C72B0')
    ax2.scatter([k], [ratios[i]], color='#4C72B0', zorder=5)
    ax2.set_ylabel('compression ratio (x)', color='#4C72B0')
    plt.tight_layout()
    plt.show()
    print(f"rank k={k}: error={errors[i]:.3f}, compression={ratios[i]:.2f}x")

widgets.interact(show_rank,
                  k=widgets.IntSlider(min=1, max=max_rank, step=1, value=2,
                                       description='rank k'));

## Exercise 3 — what did it find?

> 🇪🇸 ¿Qué encontró la descomposición por sí sola?

This is the important one.

In [ ]:
# TODO 6: Look at the first column of the hour factor matrix. At which hour is
#         it largest? Does that match what you found in TODO 2?

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
hour_factor = Us[2]                        # (24, 3) — one row per hour
peak = np.abs(hour_factor[:, 0]).argmax()
print(peak)                                # 18

print(T.sum(axis=(0, 1)).argmax())         # 18 — the same hour, from raw counts

# THE DECOMPOSITION DISCOVERED EVENING RUSH HOUR BY ITSELF. Nobody told it about
# time, traffic or commuting; it found the dominant pattern along that axis
# because that is what a decomposition does.
#
# (Take the absolute value: singular vectors are only defined up to sign, so the
# strongest component may come out negative.)

## What just happened

**4.7× fewer numbers, 6.7% error.** But the important part is TODO 6. The
strongest pattern in the hour factor peaks at **hour 18** — and that is also the
busiest hour in the raw data. The decomposition found rush hour on its own.

**Where this is used.** In tech, Tucker and CP compress the large weight tensors
inside neural networks so models run on phones instead of servers. In biotech,
applied to data such as (genes × samples × conditions), they find structure
ordinary PCA cannot reach, because **PCA can only ever see two axes**.

For real projects use [`tensorly`](https://tensorly.org), which implements both
properly. Take-home C in section 11 compares CP against what you just built.

---

## Time for Kahoot 🎯

**Kahoot 3 — Convolution & Tensor Decompositions** · 6 questions, about 5 minutes.

> 🇪🇸 **Convolución y descomposiciones tensoriales** — 6 preguntas, unos 5 minutos.

Join at **kahoot.it** with the PIN on the facilitator's screen.

- [Quiz details and facilitator notes](https://project-delphi.github.io/tensors-workshop/kahoot.html#quiz-3)
- [Import file (`.xlsx`)](https://github.com/project-delphi/tensors-workshop/blob/main/kahoot/kahoot_quiz_3_convolution_decompositions.xlsx)

Next up: **11 · Wrap-up and take-homes** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/11-wrap-up-and-take-homes.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)